In [2]:
import requests
import pandas as pd
from pathlib import Path

In [3]:
# Retrieve World Bank country metadata

url = "https://api.worldbank.org/v2/country"

params = {
    "format": "json",
    "per_page": 400
}

response = requests.get(url, params=params)
response.raise_for_status()

country_data = response.json()

country_data[1][0]

{'id': 'ABW',
 'iso2Code': 'AW',
 'name': 'Aruba',
 'region': {'id': 'LCN',
  'iso2code': 'ZJ',
  'value': 'Latin America & Caribbean '},
 'adminregion': {'id': '', 'iso2code': '', 'value': ''},
 'incomeLevel': {'id': 'HIC', 'iso2code': 'XD', 'value': 'High income'},
 'lendingType': {'id': 'LNX', 'iso2code': 'XX', 'value': 'Not classified'},
 'capitalCity': 'Oranjestad',
 'longitude': '-70.0167',
 'latitude': '12.5167'}

In [6]:
# Build country metadata table

countries = []

for item in country_data[1]:
    if item["region"]["value"] != "Aggregates":
        countries.append({
            "country_code": item["id"],
            "country_name": item["name"],
            "region": item["region"]["value"],
            "income_group": item["incomeLevel"]["value"]
        })

countries_df = pd.DataFrame(countries)

print("Number of countries:", len(countries_df))
print("Number of regions:", countries_df["region"].nunique())
print("Number of income groups:", countries_df["income_group"].nunique())

countries_df.head()

Number of countries: 217
Number of regions: 7
Number of income groups: 4


,country_code,country_name,region,income_group
0,ABW,Aruba,Latin America & Caribbean,High income
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,AGO,Angola,Sub-Saharan Africa,Lower middle income
3,ALB,Albania,Europe & Central Asia,Upper middle income
4,AND,Andorra,Europe & Central Asia,High income


In [7]:
# Development indicators used in the analysis

indicators = {
    "NY.GDP.PCAP.CD": "GDP per capita (current US$)",
    "NY.GDP.MKTP.KD.ZG": "GDP growth (annual %)",
    "SP.DYN.LE00.IN": "Life expectancy at birth",
    "SH.DYN.MORT": "Under-five mortality rate",
    "SH.XPD.CHEX.GD.ZS": "Current health expenditure (% of GDP)",
    "EG.ELC.ACCS.ZS": "Access to electricity (% of population)",
    "SP.POP.TOTL": "Population, total"
}

indicators

{'NY.GDP.PCAP.CD': 'GDP per capita (current US$)',
 'NY.GDP.MKTP.KD.ZG': 'GDP growth (annual %)',
 'SP.DYN.LE00.IN': 'Life expectancy at birth',
 'SH.DYN.MORT': 'Under-five mortality rate',
 'SH.XPD.CHEX.GD.ZS': 'Current health expenditure (% of GDP)',
 'EG.ELC.ACCS.ZS': 'Access to electricity (% of population)',
 'SP.POP.TOTL': 'Population, total'}

In [8]:
# Retrieve indicator data for 2000-2024

all_records = []

valid_country_codes = set(countries_df["country_code"])

for indicator_code in indicators:

    url = (
        f"https://api.worldbank.org/v2/country/all/"
        f"indicator/{indicator_code}"
    )

    params = {
        "format": "json",
        "date": "2000:2024",
        "per_page": 10000
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    api_data = response.json()

    if len(api_data) < 2 or api_data[1] is None:
        continue

    for item in api_data[1]:

        if item["countryiso3code"] in valid_country_codes:
            all_records.append({
                "country_code": item["countryiso3code"],
                "indicator_code": item["indicator"]["id"],
                "indicator_name": item["indicator"]["value"],
                "year": int(item["date"]),
                "value": item["value"]
            })

wdi_df = pd.DataFrame(all_records)

print("Countries:", wdi_df["country_code"].nunique())
print("Indicators:", wdi_df["indicator_code"].nunique())
print("Years:", wdi_df["year"].min(), "-", wdi_df["year"].max())
print("Observations:", len(wdi_df))

wdi_df.head()

Countries: 217
Indicators: 7
Years: 2000 - 2024
Observations: 37975


,country_code,indicator_code,indicator_name,year,value
0,AFG,NY.GDP.PCAP.CD,GDP per capita (current US$),2024,416.871146
1,AFG,NY.GDP.PCAP.CD,GDP per capita (current US$),2023,413.757895
2,AFG,NY.GDP.PCAP.CD,GDP per capita (current US$),2022,357.261153
3,AFG,NY.GDP.PCAP.CD,GDP per capita (current US$),2021,356.496214
4,AFG,NY.GDP.PCAP.CD,GDP per capita (current US$),2020,510.787063


In [10]:
# Validate retrieved WDI data

print("DATASET SUMMARY")
print("Shape:", wdi_df.shape)
print("Countries:", wdi_df["country_code"].nunique())
print("Indicators:", wdi_df["indicator_code"].nunique())
print("Period:", wdi_df["year"].min(), "-", wdi_df["year"].max())

print("\nMISSING VALUES")
print(wdi_df.isnull().sum())

# Availability by indicator
availability_by_indicator = (
    wdi_df.groupby(["indicator_code", "indicator_name"])
    .agg(
        total_records=("value", "size"),
        available_values=("value", "count")
    )
    .reset_index()
)

availability_by_indicator["missing_values"] = (
    availability_by_indicator["total_records"]
    - availability_by_indicator["available_values"]
)

print("\nDATA AVAILABILITY BY INDICATOR")
display(availability_by_indicator)

# Availability by indicator and year
availability_by_year = (
    wdi_df.groupby(["indicator_code", "indicator_name", "year"])
    .agg(
        total_records=("value", "size"),
        available_values=("value", "count")
    )
    .reset_index()
)

availability_by_year["missing_values"] = (
    availability_by_year["total_records"]
    - availability_by_year["available_values"]
)

print("\nDATA AVAILABILITY BY INDICATOR AND YEAR")
display(availability_by_year)

DATASET SUMMARY
Shape: (37975, 5)
Countries: 217
Indicators: 7
Period: 2000 - 2024

MISSING VALUES
country_code         0
indicator_code       0
indicator_name       0
year                 0
value             1903
dtype: int64

DATA AVAILABILITY BY INDICATOR


,indicator_code,indicator_name,total_records,available_values,missing_values
0,EG.ELC.ACCS.ZS,Access to electricity (% of population),5425,5350,75
1,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),5425,5168,257
2,NY.GDP.PCAP.CD,GDP per capita (current US$),5425,5242,183
3,SH.DYN.MORT,"Mortality rate, under-5 (per 1,000 live births)",5425,4900,525
4,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),5425,4562,863
5,SP.DYN.LE00.IN,"Life expectancy at birth, total (years)",5425,5425,0
6,SP.POP.TOTL,"Population, total",5425,5425,0



DATA AVAILABILITY BY INDICATOR AND YEAR


,indicator_code,indicator_name,year,total_records,available_values,missing_values
0,EG.ELC.ACCS.ZS,Access to electricity (% of population),2000,217,211,6
1,EG.ELC.ACCS.ZS,Access to electricity (% of population),2001,217,211,6
2,EG.ELC.ACCS.ZS,Access to electricity (% of population),2002,217,212,5
3,EG.ELC.ACCS.ZS,Access to electricity (% of population),2003,217,212,5
4,EG.ELC.ACCS.ZS,Access to electricity (% of population),2004,217,212,5
...,...,...,...,...,...,...
170,SP.POP.TOTL,"Population, total",2020,217,217,0
171,SP.POP.TOTL,"Population, total",2021,217,217,0
172,SP.POP.TOTL,"Population, total",2022,217,217,0
173,SP.POP.TOTL,"Population, total",2023,217,217,0


In [11]:
# Save processed datasets

project_dir = Path.cwd().parent
data_dir = project_dir / "data"

countries_df.to_csv(
    data_dir / "countries.csv",
    index=False
)

wdi_df.to_csv(
    data_dir / "wdi_indicators_2000_2024.csv",
    index=False
)

print(f"Countries saved: {len(countries_df):,}")
print(f"WDI observations saved: {len(wdi_df):,}")

Countries saved: 217
WDI observations saved: 37,975
